# 06 Property Advisory RAG

Use this notebook to combine the tuned price model with retrieved market context and generate a property-level advisory answer through Ollama.

In [1]:
import os
import sys

sys.path.append(os.path.abspath('..'))

In [2]:
from src.rag.service import ask_property_question
from src.utils.config_loader import load_yaml_config, resolve_project_path

/Volumes/NIKHILESH/Projects/real-estate-ai-advisor/real-estate-ai-platform/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
resolve_project_path('models/xgboost_price_model_tuned_clean.joblib')

PosixPath('/Volumes/NIKHILESH/Projects/real-estate-ai-advisor/real-estate-ai-platform/models/xgboost_price_model_tuned_clean.joblib')

In [4]:
rag_config = load_yaml_config('configs/rag_config.yaml')
rag_config['property_ollama_model_name'], rag_config['property_generation_top_k']

('qwen2.5-coder:7b-instruct', 3)

In [5]:
property_features = {
    'median_income': 8.3252,
    'house_age': 41.0,
    'average_rooms': 6.984127,
    'average_bedrooms': 1.02381,
    'population': 322.0,
    'average_occupancy': 2.555556,
    'latitude': 37.88,
    'longitude': -122.23,
}

In [6]:
result = ask_property_question(
    'How should I interpret this predicted price in market context?',
    property_features,
)
result['model_name'], result['predicted_price'], result['predicted_price_usd']

('qwen2.5-coder:7b-instruct', 3.959409236907959, 395940.9236907959)

In [7]:
result['answer']

"### Short Direct Answer\nThe predicted price of $395,941 is based on a model that considers various factors including income, house age, room count, and occupancy rate.\n\n### Practical Interpretation\nThis estimated price suggests that the property in question is valued at approximately $396,000. It reflects the current market conditions and the specific characteristics of the property.\n\n### Market-Context Explanation\nThe model's prediction is influenced by factors such as the median income of $83,252, which indicates a potentially strong local economy that could support higher property values. However, it's important to note that this estimate should be considered in the broader context of California's population projections and housing market dynamics.\n\n### Limitations Note\nWhile the model provides a useful estimate, it is based on historical data and may not account for future changes such as shifts in local demographics or economic conditions. It is advisable to consult wit

In [8]:
[(item['title'], item['score']) for item in result['sources']]

[('California Population Projections', 0.6473648548126221),
 ('California Population Projections', 0.5689231157302856),
 ('California Housing Snapshot', 0.560427188873291)]